In [52]:
from thinkbayes import Suite

class Cookie2(Suite):
    mixes = {
        'Bowl1':dict(vanilla=0.75, chocolate=0.25),
        'Bowl2':dict(vanilla=0.5, chocolate=0.5)
        }
    
    def Likelihood(self, data, hypo):
        mix = self.mixes[hypo]

        print(mix)
       
        total = sum(mix.values())
        print(total)
        like = mix[data] / total
        return like

hypos = ['Bowl1', 'Bowl2']
suite = Cookie2(hypos)

suite.Update('vanilla')
#suite.Update('chocolate')

suite.Items()


{'vanilla': 0.75, 'chocolate': 0.25}
1.0
{'vanilla': 0.5, 'chocolate': 0.5}
1.0


dict_items([('Bowl1', 0.6000000000000001), ('Bowl2', 0.4)])

In [53]:
class Cookie3(Suite):
    mixes = {
        'Bowl1': {'vanilla':30, 'chocolate':10},
        'Bowl2': {'vanilla':20, 'chocolate':20}
    }

    def Likelihood(self, data, hypo):
        print(data)
        mix = self.mixes[hypo]
        print(mix)
        total = sum(mix.values())
        like = mix[data] / total
        mix[data] -= 1
        return like
    
hypos = ['Bowl1', 'Bowl2']
suite = Cookie3(hypos)

suite.Update('vanilla')
suite.Update('vanilla')
print(suite.mixes)

suite.Items()

vanilla
{'vanilla': 30, 'chocolate': 10}
vanilla
{'vanilla': 20, 'chocolate': 20}
vanilla
{'vanilla': 29, 'chocolate': 10}
vanilla
{'vanilla': 19, 'chocolate': 20}
{'Bowl1': {'vanilla': 28, 'chocolate': 10}, 'Bowl2': {'vanilla': 18, 'chocolate': 20}}


dict_items([('Bowl1', 0.6960000000000001), ('Bowl2', 0.304)])

In [54]:
mix = {
    'vanilla':30,
    'chocolate':10
}

print(mix)

mix['vanilla'] -= 1

print(mix)

{'vanilla': 30, 'chocolate': 10}
{'vanilla': 29, 'chocolate': 10}


In [55]:
# -----------------------------
# 袋の中身
# -----------------------------
bowl1 = {
    "vanilla": 30,
    "chocolate": 10
}

bowl2 = {
    "vanilla": 20,
    "chocolate": 20
}

# -----------------------------
# 仮説ごとのポイント計算
# -----------------------------
def calc_score(bowl, observations):

    # 元の袋を書き換えないようコピー
    bowl = bowl.copy()

    score = 1

    for obs in observations:

        total = sum(bowl.values())

        # 今回のポイント
        score *= bowl[obs] / total

        # 食べる
        bowl[obs] -= 1

    return score

# -----------------------------
# 観測データ
# -----------------------------
observations = ["vanilla", "vanilla"]

# -----------------------------
# 各仮説のポイント
# -----------------------------
score1 = calc_score(bowl1, observations)
score2 = calc_score(bowl2, observations)

print("Bowl1 score =", score1)
print("Bowl2 score =", score2)

# -----------------------------
# ベイズ更新
# -----------------------------
prior1 = 0.5
prior2 = 0.5

post1 = prior1 * score1
post2 = prior2 * score2

total = post1 + post2

post1 /= total
post2 /= total

print()
print(f"Bowl1 = {post1:.3f}")
print(f"Bowl2 = {post2:.3f}")

Bowl1 score = 0.5576923076923077
Bowl2 score = 0.24358974358974358

Bowl1 = 0.696
Bowl2 = 0.304


In [56]:
# ==========================================
# Cookie Branching World
# ==========================================

from collections import defaultdict

# ------------------------------------------
# 初期状態
# ------------------------------------------

initial_state = (
    (30, 10),  # Bowl1 (vanilla, chocolate)
    (25, 25)   # Bowl2 (vanilla, chocolate)
)

# ------------------------------------------
# 観測
# ------------------------------------------

observations = ["vanilla", "vanilla", "vanilla"]

# ------------------------------------------
# 状態の管理
# state -> weight
# ------------------------------------------

worlds = {initial_state: 1.0}

# ------------------------------------------
# 観測ごとに世界を分岐
# ------------------------------------------

for step, obs in enumerate(observations, start=1):

    print()
    print("=" * 60)
    print(f"{step}回目の観測 : {obs}")
    print("=" * 60)

    new_worlds = defaultdict(float)

    for state, weight in worlds.items():

        (b1_v, b1_c), (b2_v, b2_c) = state

        # ----------------------------------
        # Bowl1から取り出した世界
        # ----------------------------------

        total1 = b1_v + b1_c

        if obs == "vanilla" and b1_v > 0:

            prob = b1_v / total1

            new_state = (
                (b1_v - 1, b1_c),
                (b2_v, b2_c)
            )

            new_worlds[new_state] += weight * prob

        # ----------------------------------
        # Bowl2から取り出した世界
        # ----------------------------------

        total2 = b2_v + b2_c

        if obs == "vanilla" and b2_v > 0:

            prob = b2_v / total2

            new_state = (
                (b1_v, b1_c),
                (b2_v - 1, b2_c)
            )

            new_worlds[new_state] += weight * prob

    worlds = dict(new_worlds)

    # --------------------------------------
    # 状態を表示
    # --------------------------------------

    for i, (state, weight) in enumerate(
            sorted(worlds.items(),
                   key=lambda x: x[1],
                   reverse=True),
            start=1):

        (b1_v, b1_c), (b2_v, b2_c) = state

        print(
            f"{i:2d}: "
            f"Bowl1=({b1_v:2d},{b1_c:2d}) "
            f"Bowl2=({b2_v:2d},{b2_c:2d}) "
            f"weight={weight:.6f}"
        )

# ------------------------------------------
# 最終結果
# ------------------------------------------

print()
print("=" * 60)
print("最終結果")
print("=" * 60)

total = sum(worlds.values())

for state, weight in sorted(
        worlds.items(),
        key=lambda x: x[1],
        reverse=True):

    (b1_v, b1_c), (b2_v, b2_c) = state

    print(
        f"Bowl1=({b1_v},{b1_c}) "
        f"Bowl2=({b2_v},{b2_c}) "
        f"P={weight/total:.4f}"
    )


1回目の観測 : vanilla
 1: Bowl1=(29,10) Bowl2=(25,25) weight=0.750000
 2: Bowl1=(30,10) Bowl2=(24,25) weight=0.500000

2回目の観測 : vanilla
 1: Bowl1=(29,10) Bowl2=(24,25) weight=0.750000
 2: Bowl1=(28,10) Bowl2=(25,25) weight=0.557692
 3: Bowl1=(30,10) Bowl2=(23,25) weight=0.244898

3回目の観測 : vanilla
 1: Bowl1=(28,10) Bowl2=(24,25) weight=0.836538
 2: Bowl1=(29,10) Bowl2=(23,25) weight=0.551020
 3: Bowl1=(27,10) Bowl2=(25,25) weight=0.410931
 4: Bowl1=(30,10) Bowl2=(22,25) weight=0.117347

最終結果
Bowl1=(28,10) Bowl2=(24,25) P=0.4366
Bowl1=(29,10) Bowl2=(23,25) P=0.2876
Bowl1=(27,10) Bowl2=(25,25) P=0.2145
Bowl1=(30,10) Bowl2=(22,25) P=0.0613


In [57]:
# ----------------------------------
# 初期状態
# ----------------------------------

bowls = {
    "Bowl1": {"vanilla": 30.0, "chocolate": 10.0},
    "Bowl2": {"vanilla": 25.0, "chocolate": 25.0}
}

# ----------------------------------
# 観測データ
# ----------------------------------

observations = ["vanilla", "vanilla", "vanilla"]

# ----------------------------------
# 事前確率
# ----------------------------------

posterior = {
    "Bowl1": 0.5,
    "Bowl2": 0.5
}

# ----------------------------------
# 更新
# ----------------------------------

for step, cookie in enumerate(observations, start=1):

    print()
    print("=" * 50)
    print(f"{step}回目 : {cookie}")
    print("=" * 50)

    # ------------------------------
    # ポイント計算
    # ------------------------------

    score = {}

    for bowl in posterior:

        total = sum(bowls[bowl].values())

        score[bowl] = (
            posterior[bowl]
            * bowls[bowl][cookie]
            / total
        )

    # ------------------------------
    # 正規化
    # ------------------------------

    total_score = sum(score.values())

    posterior = {
        bowl: score[bowl] / total_score
        for bowl in score
    }

    print("事後確率")

    for bowl, p in posterior.items():
        print(f"{bowl}: {p:.4f}")

    # ------------------------------
    # 分数個食べる
    # ------------------------------

    for bowl in bowls:

        bowls[bowl][cookie] -= posterior[bowl]

    print()
    print("更新後のボウル")

    for bowl, mix in bowls.items():

        print(
            bowl,
            f"V={mix['vanilla']:.3f}",
            f"C={mix['chocolate']:.3f}"
        )
        


1回目 : vanilla
事後確率
Bowl1: 0.6000
Bowl2: 0.4000

更新後のボウル
Bowl1 V=29.400 C=10.000
Bowl2 V=24.600 C=25.000

2回目 : vanilla
事後確率
Bowl1: 0.6929
Bowl2: 0.3071

更新後のボウル
Bowl1 V=28.707 C=10.000
Bowl2 V=24.293 C=25.000

3回目 : vanilla
事後確率
Bowl1: 0.7725
Bowl2: 0.2275

更新後のボウル
Bowl1 V=27.935 C=10.000
Bowl2 V=24.065 C=25.000


In [58]:
from thinkbayes import Suite


class Cookie4(Suite):

    def __init__(self, hypos):

        super().__init__(hypos)

        # クッキーの初期状態
        self.mixes = {
            'Bowl1': {
                'vanilla': 30.0,
                'chocolate': 10.0
            },
            'Bowl2': {
                'vanilla': 25.0,
                'chocolate': 25.0
            }
        }

    def Likelihood(self, data, hypo):
        """
        data : 観測したクッキーの種類
        hypo : Bowl1 または Bowl2

        この仮説なら何点かを返す
        """

        mix = self.mixes[hypo]

        total = sum(mix.values())

        point = mix[data] / total

        return point

    def EatCookie(self, color):
        """
        事後確率に応じてクッキーを食べる
        """

        for hypo, prob in self.Items():

            self.mixes[hypo][color] -= prob


# ------------------------------------
# 仮説
# ------------------------------------

hypos = ['Bowl1', 'Bowl2']

suite = Cookie4(hypos)

# ------------------------------------
# 1回目
# ------------------------------------

print("初期状態")
print(suite.mixes)

suite.Update('vanilla')

print("\n1回目更新後")

for hypo, prob in suite.Items():
    print(hypo, prob)

suite.EatCookie('vanilla')

print("\n1回目食べた後")
print(suite.mixes)

# ------------------------------------
# 2回目
# ------------------------------------

suite.Update('vanilla')

print("\n2回目更新後")

for hypo, prob in suite.Items():
    print(hypo, prob)

suite.EatCookie('vanilla')

print("\n2回目食べた後")
print(suite.mixes)


初期状態
{'Bowl1': {'vanilla': 30.0, 'chocolate': 10.0}, 'Bowl2': {'vanilla': 25.0, 'chocolate': 25.0}}

1回目更新後
Bowl1 0.6000000000000001
Bowl2 0.4

1回目食べた後
{'Bowl1': {'vanilla': 29.4, 'chocolate': 10.0}, 'Bowl2': {'vanilla': 24.6, 'chocolate': 25.0}}

2回目更新後
Bowl1 0.6929481087245771
Bowl2 0.30705189127542293

2回目食べた後
{'Bowl1': {'vanilla': 28.707051891275423, 'chocolate': 10.0}, 'Bowl2': {'vanilla': 24.292948108724577, 'chocolate': 25.0}}
